## Lecture 3:  PARSING

**Previously (L2):** Regular Expressions · Tokenization (word / char / subword: BPE, Unigram, WordPiece, SentencePiece) · Normalization (stemming, lemmatization)

**Outline:**
- What is parsing and why
- Terminology: grammar, syntax, POS tags, constituency, CFG
- **Syntactic parsing**
  - Constituency parsing: top-down, bottom-up, CKY, ambiguity, PCFG
  - Dependency parsing: transition-based, graph-based
  - Methods: rule-based, statistical, neural; parsing in Python (spaCy)
  - Shallow parsing / chunking
- **Semantic parsing**

#### Quiz cheat sheet (regex recap from L2)
| Special chars | | Quantifiers | |
|---|---|---|---|
| `\` | escape special chars | `*` | 0 or more (append `?` for non-greedy) |
| `.` | any character | `+` | 1 or more (append `?` for non-greedy) |
| `^` | beginning of string | `?` | 0 or 1 (append `?` for non-greedy) |
| `$` | end of string | `{m}` | exactly m |
| `[5b-d]` | any of '5', 'b', 'c', 'd' | `{m,n}` | m to n (m defaults to 0, n to infinity) |
| `[^a-c6]` | any char except 'a', 'b', 'c', '6' | `{m,n}?` | m to n, as few as possible |
| `R\|S` | regex R or regex S | `\b`, `\d`, `\s`, `\w` | boundary, digit, whitespace, word char |
| `()` | capture group | `\D`, `\S`, `\W` | negations |

Example: `r"[Hh]ello(!)?"` matches `Hello`, `Hello!`, `hello`, `hello!`

In [7]:
import re
for s in ['Hello', 'Hello!', 'hello', 'hello!', 'HeLlO', 'Hello2']:
    print(f'{s:8}', bool(re.fullmatch(r"[Hh]ello(!)?", s)))

Hello    True
Hello!   True
hello    True
hello!   True
HeLlO    False
Hello2   False


# Parsing

**Definition:** the process of analyzing the structure of a text, both in terms of its **grammar** (syntactic parsing) and its **meaning** (semantic parsing).

Break a sentence down into its constituent elements and understand their **syntactic roles**, **relationships**, and **semantic implications**.
e.g. *"The patient complained of abdominal pain."*

### Why?
- **Machine translation**: word order and structure differ between languages; the parse tells you how to reorder.
- **Text summarization**: knowing the main clause vs. modifiers helps decide what to keep.

### Parsing vs. Tokenization
Tokenization breaks text into pieces; parsing figures out **how those pieces relate** to each other.

| Aspect | Tokenization | Parsing |
|---|---|---|
| Purpose | Split text | Understand structure |
| Output | List of tokens | Tree or graph |
| Grammar-aware | No | Yes |
| Typical position | Very early | Later in pipeline |
| Complexity | Low | Higher |
| Example question | "What are the words?" | "Who did what to whom?" |

### Types of parsing
- **Syntactic parsing**: analyze a string of words according to the rules of a formal grammar to determine its grammatical structure (how words group into phrases and how phrases relate hierarchically).
  - **Constituency parsing**: a *constituent* is a group of words (or one word) that functions as a single unit in the sentence structure.
  - **Dependency parsing**: directed relations between words.
- **Semantic parsing**: goes beyond syntax to extract the sentence's **meaning**; understand the roles of words in the context of a task and how they interact.

# Terminology
Grammar · Syntax · Part-of-Speech (POS) tags · Constituency · Constituency Grammar · Context-Free Grammar (CFG)

### Syntax and grammar, an example
*"The quick brown fox jumps over the lazy dog"*
- Nouns: fox, dog · Verbs: jumps · Adjectives: quick, brown, lazy · Articles: the
- "The quick brown fox" → **Subject** · "jumps" → **Verb** · "over the lazy dog" → **Adverbial phrase**

### Grammar
The set of rules that govern the composition of phrases, clauses, and words in a natural language. It covers:
- **Phonology**: sound patterns (spoken language)
- **Morphology**: how words are formed from smaller units (e.g. adding "-ed" for past tense)
- **Syntax**: how words are arranged into phrases and sentences
- **Semantics**: how meaning is constructed and interpreted

### Syntax
The arrangement of words and phrases to create well-formed sentences, as **rules** dictating how words combine into phrases and sentences.

Syntactic categories and common denotations:

| Symbol | Meaning | Symbol | Meaning |
|---|---|---|---|
| S | sentence | TV | transitive verb (takes an object) |
| NP | noun phrase | IV | intransitive verb |
| VP | verb phrase | PREP | preposition |
| DET | determiner (article) | PP | prepositional phrase |
| N / NN | noun | ADJ | adjective |

The word-level categories (DET, N, TV, IV, PREP, ADJ) are **Part-of-Speech (POS) tags**.

## Part-of-Speech (POS) Tagging
The process of assigning grammatical categories to words to identify their roles and functions in a sentence.

### Penn Treebank tagset (most common tags)
| Tag | Description | Example | Tag | Description | Example |
|---|---|---|---|---|---|
| CC | coordinating conjunction | and, but, or | NNP | proper noun, singular | IBM |
| CD | cardinal number | one, two | NNPS | proper noun, plural | Carolinas |
| DT | determiner | a, the | PRP | personal pronoun | I, you, he |
| EX | existential 'there' | there | PRP\$ | possessive pronoun | your, one's |
| FW | foreign word | mea culpa | RB | adverb | quickly, never |
| IN | preposition / sub-conj | of, in, by | RBR / RBS | adverb comparative / superlative | faster / fastest |
| JJ | adjective | yellow | RP | particle | up, off |
| JJR / JJS | adj. comparative / superlative | bigger / wildest | TO | "to" | to |
| LS | list item marker | 1, 2, One | UH | interjection | ah, oops |
| MD | modal | can, should | VB | verb, base form | eat |
| NN | noun, singular or mass | llama | VBD | verb, past tense | ate |
| NNS | noun, plural | llamas | VBG | verb, gerund | eating |
| PDT | predeterminer | all, both | VBN | verb, past participle | eaten |
| POS | possessive ending | 's | VBP / VBZ | verb non-3sg / 3sg present | eat / eats |
| WDT | wh-determiner | which, that | WP / WP\$ / WRB | wh-pronoun / possessive / adverb | what / whose / how |

### POS ambiguity
Words may have more than one POS tag. Example **"back"**:
- The **back** door → JJ
- On my **back** → NN
- Win the voters **back** → RB
- Promised to **back** the bill → VB

In [8]:
# POS tagging with NLTK (uses the Penn Treebank tagset)
# !pip install nltk
import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True); nltk.download('averaged_perceptron_tagger_eng', quiet=True)

for s in ["The back door", "On my back", "Win the voters back", "Promised to back the bill"]:
    print(nltk.pos_tag(nltk.word_tokenize(s)))

[('The', 'DT'), ('back', 'JJ'), ('door', 'NN')]
[('On', 'IN'), ('my', 'PRP$'), ('back', 'NN')]
[('Win', 'IN'), ('the', 'DT'), ('voters', 'NNS'), ('back', 'RB')]
[('Promised', 'VBN'), ('to', 'TO'), ('back', 'VB'), ('the', 'DT'), ('bill', 'NN')]


## Constituency
**Idea:** groups of words can behave as single units, called **constituents**.
They can be grouped *because* they can all appear in similar syntactic environments (e.g. "a flight", "the morning flight", "Houston" can all appear before "leaves at 9").

### Constituency trees
- **Internal nodes** correspond to phrases: S (sentence), NP (noun phrase: *a flight*), VP (verb phrase: *landed*), PP (prepositional phrase: *with a friend*)
- **Nodes immediately above words** are POS tags: PN (pronoun), Det (determiner), V (verb), N (noun), P (preposition)

```
      NP
     /  \
   Det   Nom
    |     |
    a    Noun
          |
        flight
```

## Grammar types
- **Constituency grammar**: views a sentence's structure as an arrangement of subparts (constituents), organized hierarchically in a tree.
- **Context-free grammar (CFG)**: a formal grammar of rules for combining smaller grammatical units into larger ones, based on the syntactic **categories** of the units, not their specific context in the sentence.
- **Dependency grammar**: based on **dependency relations between words**.

CFGs can be used to generate constituency trees, but **not all** constituency trees can be generated by CFGs.

# Context-Free Grammar (CFG)
- Also known as **phrase-structure grammars**.
- A set of **rules** (productions), each expressing how symbols of the language can be grouped and ordered together.

**Terminology:**
- **Root**: represents the entire grammatical unit (start symbol, usually S)
- **Terminal**: symbols that correspond to words
- **Non-terminal**: symbols that express abstractions over terminals (NP, VP, Det, …)
- **Derivation**: a sequence of rule expansions

### Building up the rules
| Rule | Meaning | Example |
|---|---|---|
| `NP → Det Nominal` | rewrite NP as a determiner* + nominal** | *a flight* |
| `Nominal → Noun` | a nominal can be just a noun | *flight* |
| `S → NP VP` | a sentence is an NP followed by a VP | *I* / *prefer a morning flight* |
| `VP → Verb NP` | a verb followed by assorted other things | *prefer a morning flight* |
| `VP → Verb NP PP` | a verb followed by NP and PP*** | *leave Boston in the morning* |
| `VP → Verb PP` | a verb followed by a PP alone | *leaving on Thursday* |
| `PP → Preposition NP` | preposition followed by NP | *from Los Angeles* |

\* **Det**: word used at the beginning of a noun group (a, the, some, this, each)
\** **Nominal**: words that function together as a noun
\*** **PP**: one preposition plus the object it governs

### Bracketed representation
`[S [NP [Pro I]] [VP [V prefer] [NP [Det a] [Nom [Nom [N morning]] [N flight]]]]]`

### Formal definition: a 4-tuple G = (N, Σ, R, S)
| | |
|---|---|
| **N** | a set of **non-terminal** symbols (variables) |
| **Σ** | a set of **terminal** symbols (disjoint from N) |
| **R** | a set of rules `A → β`, where A is a non-terminal and β is a string from (Σ ∪ N)* |
| **S** | a designated **start symbol**, a member of N |

### Grammar vs. Lexicon
- **Grammar**: production rules using non-terminals to show structure, e.g. `S → NP VP`
- **Lexicon**: rules that lead directly to a **terminal** word, e.g. `Det → "the" | "a"`

**Derivation:** the sequence of productions used to generate a string of words, usually visualized as a **parse tree**.

### Why "context-free"?
The rules apply **regardless of the surrounding symbols or context**. `NP → Det N` can be applied to any NP, whether it is the subject or the object.
```
S → NP VP        Det → "the" | "a"
NP → Det N       N → "cat" | "dog" | "fish"
VP → V NP        V → "eats" | "chases"
```
→ *"The fish eats the dog."*

### The L1 miniature English grammar (Jurafsky & Martin, Fig. 17.8)
Used throughout the parsing examples below.

| Grammar | Lexicon |
|---|---|
| S → NP VP \| Aux NP VP \| VP | Det → that \| this \| the \| a |
| NP → Pronoun \| Proper-Noun \| Det Nominal | Noun → book \| flight \| meal \| money |
| Nominal → Noun \| Nominal Noun \| Nominal PP | Verb → book \| include \| prefer |
| VP → Verb \| Verb NP \| Verb NP PP \| Verb PP \| VP PP | Pronoun → I \| she \| me |
| PP → Preposition NP | Proper-Noun → Houston \| NWA |
| | Aux → does |
| | Preposition → from \| to \| on \| near \| through |

In [9]:
import random

# A CFG as a dict: non-terminal -> list of right-hand sides (tuples). Terminals are lowercase strings.
L1_GRAMMAR = {
    'S':       [('NP', 'VP'), ('Aux', 'NP', 'VP'), ('VP',)],
    'NP':      [('Pronoun',), ('Proper-Noun',), ('Det', 'Nominal')],
    'Nominal': [('Noun',), ('Nominal', 'Noun'), ('Nominal', 'PP')],
    'VP':      [('Verb',), ('Verb', 'NP'), ('Verb', 'NP', 'PP'), ('Verb', 'PP'), ('VP', 'PP')],
    'PP':      [('Preposition', 'NP')],
}
L1_LEXICON = {
    'Det': ['that', 'this', 'the', 'a'],
    'Noun': ['book', 'flight', 'meal', 'money'],
    'Verb': ['book', 'include', 'prefer'],
    'Pronoun': ['I', 'she', 'me'],
    'Proper-Noun': ['Houston', 'NWA'],
    'Aux': ['does'],
    'Preposition': ['from', 'to', 'on', 'near', 'through'],
}

def generate(symbol='S', depth=0):
    """Random derivation: keep rewriting non-terminals until only words are left."""
    if symbol in L1_LEXICON:
        return [random.choice(L1_LEXICON[symbol])]
    rules = L1_GRAMMAR[symbol]
    rhs = rules[0] if depth > 4 else random.choice(rules)   # stop recursion getting too deep
    return [w for s in rhs for w in generate(s, depth + 1)]

# Every output is grammatical according to the CFG, even when it is nonsense:
# a CFG captures syntax, not meaning.
random.seed(1)
for _ in range(5):
    print(' '.join(generate()))

that money money prefer to she
NWA prefer on this book on I
me include near that flight near she
include me
NWA include


### Exercise: derive *"The man read this book"*
```
S → NP VP
  → Det NOM VP
  → The NOM VP
  → The Noun VP
  → The man VP
  → The man Verb NP
  → The man read NP
  → The man read Det NOM
  → The man read this NOM
  → The man read this Noun
  → The man read this book
```
This is a **leftmost derivation** (always expand the leftmost non-terminal). The resulting parse tree:
`[S [NP [Det The] [NOM [Noun man]]] [VP [Verb read] [NP [Det this] [NOM [Noun book]]]]]`

In [10]:
def leftmost_derivation(tree):
    """Print every step of a leftmost derivation from a tree written as nested tuples (label, children...)."""
    def is_leaf(t): return isinstance(t, str)
    frontier = [tree]
    print(' '.join(t if is_leaf(t) else t[0] for t in frontier))
    while any(not is_leaf(t) for t in frontier):
        i = next(k for k, t in enumerate(frontier) if not is_leaf(t))
        frontier = frontier[:i] + list(frontier[i][1:]) + frontier[i+1:]
        print('→', ' '.join(t if is_leaf(t) else t[0] for t in frontier))

tree = ('S', ('NP', ('Det', 'The'), ('NOM', ('Noun', 'man'))),
             ('VP', ('Verb', 'read'), ('NP', ('Det', 'this'), ('NOM', ('Noun', 'book')))))
leftmost_derivation(tree)

S
→ NP VP
→ Det NOM VP
→ The NOM VP
→ The Noun VP
→ The man VP
→ The man Verb NP
→ The man read NP
→ The man read Det NOM
→ The man read this NOM
→ The man read this Noun
→ The man read this book


In [11]:
def pretty(tree, indent=''):
    """Print a nested-tuple tree sideways."""
    if isinstance(tree, str):
        print(indent + tree); return
    label, *kids = tree
    if len(kids) == 1 and isinstance(kids[0], str):
        print(f'{indent}{label}: {kids[0]}'); return
    print(indent + label)
    for k in kids:
        pretty(k, indent + '    ')

def bracket(tree):
    if isinstance(tree, str): return tree
    return '[' + tree[0] + ' ' + ' '.join(bracket(k) for k in tree[1:]) + ']'

pretty(tree)
print(bracket(tree))

S
    NP
        Det: The
        NOM
            Noun: man
    VP
        Verb: read
        NP
            Det: this
            NOM
                Noun: book
[S [NP [Det The] [NOM [Noun man]]] [VP [Verb read] [NP [Det this] [NOM [Noun book]]]]]


In [12]:
# The same with NLTK: define a CFG and parse
from nltk import CFG
from nltk.parse import ChartParser

grammar = CFG.fromstring("""
S -> NP VP
NP -> Det NOM
NOM -> Noun
VP -> Verb NP
Det -> 'The' | 'this'
Noun -> 'man' | 'book'
Verb -> 'read'
""")
for t in ChartParser(grammar).parse("The man read this book".split()):
    print(t); t.pretty_print()

(S
  (NP (Det The) (NOM (Noun man)))
  (VP (Verb read) (NP (Det this) (NOM (Noun book)))))
              S                
      ________|____             
     |             VP          
     |         ____|____        
     NP       |         NP     
  ___|___     |     ____|___    
 |      NOM   |    |       NOM 
 |       |    |    |        |   
Det     Noun Verb Det      Noun
 |       |    |    |        |   
The     man  read this     book



# Constituency Parsing
To a computer, a sentence is just a long string of characters. **Parsing** analyzes that string to determine whether it conforms to the rules of a specific grammar.

| | |
|---|---|
| **Input** | a sequence of tokens from the lexicon ("The", "robot", "paints") |
| **Processor** | the parser |
| **Reference** | the grammar |
| **Output** | a structured representation, usually a **parse tree** |

**Strategies:** Top-Down and Bottom-Up.

## Top-Down Parsing (goal-driven)
1. Start with the highest-level non-terminal, **S** (the whole sentence).
2. Find the tops of all trees that can start with S → look for rules with S on the left side (`S → NP VP`, `S → Aux NP VP`, `S → VP`).
3. Expand the constituents of each of those trees in the same way.
4. Repeat until reaching terminal nodes (POS tags → words).
5. **Reject** trees whose leaves fail to match ALL the words in the sentence.

**Example: "Book that flight."** Only `S → VP` survives, because "book" can't be a Pronoun/Proper-Noun (NP) or an Aux. Then `VP → Verb NP`, `NP → Det Nominal`, `Nominal → Noun`:

`[S [VP [Verb book] [NP [Det that] [Nominal [Noun flight]]]]]`

⚠ A naive top-down parser **loops forever** on left-recursive rules such as `Nominal → Nominal Noun` or `VP → VP PP`. The version below avoids that by always parsing over a fixed span of words, which gets strictly smaller on each recursive call.

In [13]:
def lexicon_tags(word):
    return [pos for pos, words in L1_LEXICON.items() if word.lower() in [w.lower() for w in words]]

def top_down(symbol, words, i, j, trace=False, depth=0):
    """All parse trees for `symbol` covering words[i:j], expanding rules from the top."""
    if trace and depth <= 1:
        print('  ' * depth + f'try {symbol} over {words[i:j]}')
    trees = []
    if symbol in L1_LEXICON:                                 # POS -> word
        if j == i + 1 and symbol in lexicon_tags(words[i]):
            trees.append((symbol, words[i]))
        return trees
    for rhs in L1_GRAMMAR[symbol]:
        for kids in split(rhs, words, i, j, trace, depth):
            trees.append((symbol, *kids))
    return trees

def split(rhs, words, i, j, trace, depth):
    """Try every way to divide words[i:j] among the symbols of rhs (each symbol takes >= 1 word)."""
    if len(rhs) == 1:
        # unary rule: the child covers the same span
        yield from ([t] for t in top_down(rhs[0], words, i, j, trace, depth + 1))
        return
    for k in range(i + 1, j - len(rhs) + 2):
        for first in top_down(rhs[0], words, i, k, trace, depth + 1):
            for rest in split(rhs[1:], words, k, j, trace, depth):
                yield [first] + rest

words = "Book that flight".split()
parses = top_down('S', words, 0, len(words), trace=True)
print(f'\n{len(parses)} parse(s):')
for p in parses:
    print(bracket(p))

try S over ['Book', 'that', 'flight']
  try NP over ['Book']
  try NP over ['Book', 'that']
  try Aux over ['Book']
  try VP over ['Book', 'that', 'flight']

1 parse(s):
[S [VP [Verb Book] [NP [Det that] [Nominal [Noun flight]]]]]


## Bottom-Up Parsing (data-driven)
1. Start with the **individual words** of the sentence.
2. Work upward, applying production rules to combine words into larger constituents.
3. Look for places where the **right-hand side** of a rule matches.
4. **Success** = reaching the start symbol **S** covering the whole sentence.

**Example: "Book that flight."**
1. Words → POS: *book* = Noun **or** Verb, *that* = Det, *flight* = Noun
2. Noun → Nominal; Det Nominal → NP
3. Path A: *book* as Noun → Nominal … but `Nominal NP` doesn't match any rule → **dead end (?)**
4. Path B: *book* as Verb; Verb NP → VP; VP → S ✅

Top-down wastes effort on trees that can't match the words; bottom-up wastes effort on constituents that can never become an S. **CKY** fixes the bottom-up waste of recomputation with dynamic programming.

In [6]:
# Bottom-up, step 1: which POS tags can each word take?
for w in "Book that flight".split():
    print(f'{w:8} -> {lexicon_tags(w)}')

Book     -> ['Noun', 'Verb']
that     -> ['Det']
flight   -> ['Noun']


# CYK / CKY Algorithm
**Cocke–Younger–Kasami**
- A **bottom-up** approach.
- **Dynamic programming**: partial solutions are stored and reused to find all possible parses of the sentence.
- Requires the grammar to be in **Chomsky Normal Form (CNF)**.

## Chomsky Normal Form (CNF)
Every rule `A → β` must be one of:
1. `A → B C` (exactly two non-terminals)
2. `A → s` (a single terminal)

(Named after **Noam Chomsky** (b. Dec 7, 1928), the "father of modern linguistics".)

### Converting to CNF
| Problem | Example | Rewrite |
|---|---|---|
| more than 2 non-terminals | `A → B C D` | `X1 → B C`, `A → X1 D` |
| mix of terminals and non-terminals | `A → s B` | `X1 → s`, `A → X1 B` |
| single non-terminal (unit rule) | `A → B`, `B → C` | `A → C` |

### Example (adapted from David Bamman)
Before:
```
S → NP VP               DET → "an" | "my"
VP → VP PP              VBD → "shot"
VP → VBD NP             NNS → "pajamas"
Nominal → Nominal PP    NN  → "elephant"
Nominal → NN            NP  → "I"
Nominal → NNS           PRP → "I"
PP → IN NP              IN  → "in"
NP → DET NP             PRP$ → "my"
NP → Nominal
NP → PRP$ Nominal
```
Removing the unit rules `Nominal → NN`, `Nominal → NNS`, `NP → Nominal` by pushing the words up:
```
Nominal → "pajamas" | "elephant"
NP → "elephant" | "I"
```

In [7]:
from itertools import count

def to_cnf(grammar, lexicon):
    """Convert a CFG (dict NT -> list of RHS tuples) + lexicon to CNF: binarize and remove unit rules."""
    fresh = (f'X{i}' for i in count(1))
    rules = [(lhs, rhs) for lhs, rhss in grammar.items() for rhs in rhss]
    lex = {(pos, w) for pos, ws in lexicon.items() for w in ws}

    # 1. binarize long rules: A -> B C D  =>  X1 -> B C, A -> X1 D
    binary = []
    for lhs, rhs in rules:
        while len(rhs) > 2:
            x = next(fresh); binary.append((x, rhs[:2])); rhs = (x,) + rhs[2:]
        binary.append((lhs, rhs))

    # 2. remove unit rules A -> B by copying B's rules up to A
    units = {(a, r[0]) for a, r in binary if len(r) == 1}
    closure = set(units)
    while True:
        new = {(a, c) for a, b in closure for b2, c in closure if b == b2} - closure
        if not new: break
        closure |= new
    final = [(a, r) for a, r in binary if len(r) == 2]
    final_lex = set(lex)
    for a, b in closure:
        final += [(a, r) for b2, r in binary if b2 == b and len(r) == 2]
        final_lex |= {(a, w) for b2, w in lex if b2 == b}
    return sorted(set(final)), sorted(final_lex)

g = {'S': [('NP', 'VP')], 'VP': [('VP', 'PP'), ('VBD', 'NP')], 'Nominal': [('Nominal', 'PP'), ('NN',), ('NNS',)],
     'PP': [('IN', 'NP')], 'NP': [('DET', 'NP'), ('Nominal',), ('PRP$', 'Nominal')]}
lx = {'DET': ['an', 'my'], 'VBD': ['shot'], 'NNS': ['pajamas'], 'NN': ['elephant'], 'NP': ['I'], 'PRP': ['I'], 'IN': ['in'], 'PRP$': ['my']}
rules, lexical = to_cnf(g, lx)
for a, r in rules: print(f'{a} → {" ".join(r)}')
print()
for a, w in lexical: print(f'{a} → "{w}"')

NP → DET NP
NP → Nominal PP
NP → PRP$ Nominal
Nominal → Nominal PP
PP → IN NP
S → NP VP
VP → VBD NP
VP → VP PP

DET → "an"
DET → "my"
IN → "in"
NN → "elephant"
NNS → "pajamas"
NP → "I"
NP → "elephant"
NP → "pajamas"
Nominal → "elephant"
Nominal → "pajamas"
PRP → "I"
PRP$ → "my"
VBD → "shot"


## CKY parsing: bottom-up dynamic programming
Grammar used for **"I shot an elephant in my pajamas"** (unit rules `Nominal → N`, `NP → PRP` kept here for readability; the code below handles them):
```
S → NP VP               DET → "an"
VP → VP PP              VBD → "shot"
VP → VBD NP             N   → "elephant" | "pajamas"
Nominal → Nominal PP    PRP → "I"
Nominal → N             IN  → "in"
PP → IN NP              PRP$ → "my"
NP → DET Nominal
NP → PRP$ Nominal
NP → PRP
```
**How the chart is filled:**
- Cell `[i, j]` holds every non-terminal that can span words `i..j-1`.
- **Diagonal** (span 1): look up each word's POS tags in the lexicon (plus unit rules: `PRP → NP`, `N → Nominal`).
- **Longer spans**, filled bottom-up (shortest first): for every split point `k`, if `B ∈ [i,k]` and `C ∈ [k,j]` and `A → B C` is a rule, add `A` to `[i,j]`.
  - e.g. `[PRP][VBD]` → is there a rule `? → PRP VBD`? No → the cell stays empty.
  - `[DET][N]` → `NP → DET Nominal` → "an elephant" is an **NP**.
- The sentence is grammatical if **S ∈ [0, n]**. Backpointers let you rebuild **every** parse.

For this sentence, the top cell holds **two** S analyses (S1 via `VP → VBD NP` with the PP inside the NP, S2 via `VP → VP PP`), i.e. two parse trees.

In [8]:
from collections import defaultdict

CYK_RULES = [('S', 'NP', 'VP'), ('VP', 'VP', 'PP'), ('VP', 'VBD', 'NP'), ('Nominal', 'Nominal', 'PP'),
             ('PP', 'IN', 'NP'), ('NP', 'DET', 'Nominal'), ('NP', 'PRP$', 'Nominal')]
CYK_UNARY = [('Nominal', 'N'), ('NP', 'PRP')]
CYK_LEX = {'an': ['DET'], 'shot': ['VBD'], 'elephant': ['N'], 'pajamas': ['N'], 'I': ['PRP'], 'in': ['IN'], 'my': ['PRP$']}

def add_unaries(cell):
    changed = True
    while changed:
        changed = False
        for a, b in CYK_UNARY:
            if b in cell:
                for t in list(cell[b]):
                    if (a, t) not in cell.get(a, []):
                        cell.setdefault(a, []).append((a, t)); changed = True

def cky(words):
    n = len(words)
    chart = [[{} for _ in range(n + 1)] for _ in range(n + 1)]     # chart[i][j] = {label: [trees]}
    for i, w in enumerate(words):                                  # diagonal: lexicon
        for pos in CYK_LEX.get(w, []):
            chart[i][i+1].setdefault(pos, []).append((pos, w))
        add_unaries(chart[i][i+1])
    for span in range(2, n + 1):                                   # longer spans, bottom-up
        for i in range(n - span + 1):
            j = i + span
            for k in range(i + 1, j):                              # every split point
                for a, b, c in CYK_RULES:
                    for tb in chart[i][k].get(b, []):
                        for tc in chart[k][j].get(c, []):
                            chart[i][j].setdefault(a, []).append((a, tb, tc))
            add_unaries(chart[i][j])
    return chart

words = "I shot an elephant in my pajamas".split()
chart = cky(words)

# print the chart (upper triangle): row i = start word, column j = end word
w = 14
print(' ' * 10 + ''.join(f'{x:^{w}}' for x in words))
for i in range(len(words)):
    row = ''
    for j in range(1, len(words) + 1):
        labels = ','.join(sorted(set(chart[i][j]))) if j > i else ''
        row += f'{labels or ("." if j > i else ""):^{w}}'
    print(f'{words[i]:10}{row}')

                I            shot           an         elephant         in            my         pajamas    
I             NP,PRP          .             .             S             .             .             S       
shot                         VBD            .             VP            .             .             VP      
an                                         DET            NP            .             .             NP      
elephant                                              N,Nominal         .             .          Nominal    
in                                                                      IN            .             PP      
my                                                                                   PRP$           NP      
pajamas                                                                                         N,Nominal   


In [9]:
parses = chart[0][len(words)].get('S', [])
print(f'Grammatical? {bool(parses)}   Number of parses: {len(parses)}\n')
for p in parses:
    print(bracket(p)); print()

Grammatical? True   Number of parses: 2

[S [NP [PRP I]] [VP [VBD shot] [NP [DET an] [Nominal [Nominal [N elephant]] [PP [IN in] [NP [PRP$ my] [Nominal [N pajamas]]]]]]]]

[S [NP [PRP I]] [VP [VP [VBD shot] [NP [DET an] [Nominal [N elephant]]]] [PP [IN in] [NP [PRP$ my] [Nominal [N pajamas]]]]]]



### CKY: benefits and limits
- ✅ **Verifies** whether a sentence is grammatical in the language defined by the CFG.
- ✅ Builds **all possible parses** of a sentence for a given CFG.
- ❌ It does **NOT** resolve structural ambiguity; it can't tell you which parse is **most likely**.

## CFG Ambiguity
If a CFG **G** has more than one derivation tree for some string *w ∈ L(G)*, G is an **ambiguous grammar**.

*"I saw the man with the telescope"*
- Interpretation 1: the speaker **used a telescope** to see the man → PP attaches to the **VP**
- Interpretation 2: the speaker saw a man who **had a telescope** → PP attaches to the **NP**

### Types of ambiguity
| Type | Definition | Example |
|---|---|---|
| **Attachment** | a constituent can be attached to the tree at more than one place | "I shot an elephant in my pajamas" (who is in the pajamas?) |
| **Coordination** | phrases conjoined by a conjunction like *and* | "the old men and women" → [old [men and women]] vs [old men] and [women] |
| **Scope** | unclear how far a modifier or operator extends | "He didn't speak to her or acknowledge her" → neither spoke nor acknowledged, vs didn't speak but did acknowledge |

# Probabilistic Context-Free Grammar (PCFG)
**Dealing with ambiguity:** assign a **probability to each production rule**, then choose the **most likely** interpretation.
- Probabilities reflect how frequently a production is used in real-world text.
- For each non-terminal, its rules' probabilities sum to 1.
- **P(tree)** = product of the probabilities of all rules used in the tree.

### Learning the probabilities
From a **treebank** (a collection of sentences annotated with constituency trees), use the **maximum likelihood estimate**:

$$P(X \to \alpha) = \frac{C(X \to \alpha)}{C(X)} = \frac{\text{\# times the rule was used in the corpus}}{\text{\# times non-terminal X appeared in the treebank}}$$

**Without a treebank:** the **inside-outside algorithm** (an EM algorithm):
1. Randomly initialize probabilities
2. Compute parses
3. Compute expected counts for rules
4. Re-estimate probabilities
5. Repeat

### Penn Treebank Project
- A large annotated corpus of English with grammatical structure and parse trees.
- Developed late 1980s / early 1990s by University of Pennsylvania linguists.
- Over **4.5 million words** of American English annotated with POS tags and parse trees.
- Text from many domains: news, transcripts, academic papers, etc.
- Enabled **statistical parsing** by providing training data; became a standard NLP dataset (parsing, POS tagging, language modeling) and inspired treebanks for other languages.

In [10]:
from collections import Counter

# MLE rule probabilities from a toy treebank of nested-tuple trees
treebank = [
    ('S', ('NP', ('Det', 'the'), ('N', 'dog')), ('VP', ('V', 'barks'))),
    ('S', ('NP', ('Det', 'a'), ('N', 'cat')), ('VP', ('V', 'sees'), ('NP', ('Det', 'the'), ('N', 'dog')))),
    ('S', ('NP', ('PN', 'John')), ('VP', ('V', 'sees'), ('NP', ('PN', 'Mary')))),
]

def rules_of(t):
    if isinstance(t, str): return []
    lhs, *kids = t
    rhs = tuple(k if isinstance(k, str) else k[0] for k in kids)
    return [(lhs, rhs)] + [r for k in kids for r in rules_of(k)]

rule_c = Counter(r for t in treebank for r in rules_of(t))
lhs_c = Counter()
for (lhs, _), c in rule_c.items(): lhs_c[lhs] += c

for (lhs, rhs), c in sorted(rule_c.items()):
    print(f'P({lhs} → {" ".join(rhs):8}) = {c}/{lhs_c[lhs]} = {c/lhs_c[lhs]:.2f}')

P(Det → a       ) = 1/3 = 0.33
P(Det → the     ) = 2/3 = 0.67
P(N → cat     ) = 1/3 = 0.33
P(N → dog     ) = 2/3 = 0.67
P(NP → Det N   ) = 3/5 = 0.60
P(NP → PN      ) = 2/5 = 0.40
P(PN → John    ) = 1/2 = 0.50
P(PN → Mary    ) = 1/2 = 0.50
P(S → NP VP   ) = 3/3 = 1.00
P(V → barks   ) = 1/3 = 0.33
P(V → sees    ) = 2/3 = 0.67
P(VP → V       ) = 1/3 = 0.33
P(VP → V NP    ) = 2/3 = 0.67


### Example: *"Astronomers saw stars with ears"* (Manning & Schütze)
| Rule | P | Rule | P |
|---|---|---|---|
| S → NP VP | 1.0 | NP → NP PP | 0.4 |
| PP → P NP | 1.0 | NP → astronomers | 0.1 |
| VP → V NP | 0.7 | NP → ears | 0.18 |
| VP → VP PP | 0.3 | NP → saw | 0.04 |
| P → with | 1.0 | NP → stars | 0.18 |
| V → saw | 1.0 | NP → telescopes | 0.1 |

- **t1** (PP attaches to NP, *stars with ears*): P = 1.0 × 0.1 × 0.7 × 1.0 × 0.4 × 0.18 × 1.0 × 1.0 × 0.18 = **0.0009072**
- **t2** (PP attaches to VP, *saw … with ears*): P = 1.0 × 0.1 × 0.3 × 0.7 × 1.0 × 0.18 × 1.0 × 1.0 × 0.18 = **0.0006804**

→ The PCFG picks **t1**.

### Probabilistic CKY (Jurafsky & Martin, Fig. 14.3)
Same as CKY, but each cell keeps, for each non-terminal, only the **maximum probability** and a **backpointer**:
```
function PROBABILISTIC-CKY(words, grammar) returns most probable parse and its probability
  for j ← 1 to LENGTH(words):
      for all {A | A → words[j] ∈ grammar}:  table[j-1, j, A] ← P(A → words[j])
      for i ← j-2 downto 0:
          for k ← i+1 to j-1:
              for all {A | A → B C ∈ grammar, table[i,k,B] > 0, table[k,j,C] > 0}:
                  if table[i,j,A] < P(A → B C) × table[i,k,B] × table[k,j,C]:
                      table[i,j,A] ← P(A → B C) × table[i,k,B] × table[k,j,C]
                      back[i,j,A] ← {k, B, C}
  return BUILD-TREE(back[1, LENGTH(words), S]), table[1, LENGTH(words), S]
```

In [11]:
PCFG_BIN = [('S', 'NP', 'VP', 1.0), ('PP', 'P', 'NP', 1.0), ('VP', 'V', 'NP', 0.7),
            ('VP', 'VP', 'PP', 0.3), ('NP', 'NP', 'PP', 0.4)]
PCFG_LEX = {('P', 'with'): 1.0, ('V', 'saw'): 1.0, ('NP', 'astronomers'): 0.1, ('NP', 'ears'): 0.18,
            ('NP', 'saw'): 0.04, ('NP', 'stars'): 0.18, ('NP', 'telescopes'): 0.1}

def probabilistic_cky(words, keep_all=False):
    n = len(words)
    table = defaultdict(dict)            # table[(i,j)][A] = list of (prob, tree)
    for j in range(1, n + 1):
        for (a, w), p in PCFG_LEX.items():
            if w == words[j-1]:
                table[(j-1, j)].setdefault(a, []).append((p, (a, w)))
        for i in range(j - 2, -1, -1):
            for k in range(i + 1, j):
                for a, b, c, p in PCFG_BIN:
                    for pb, tb in table[(i, k)].get(b, []):
                        for pc, tc in table[(k, j)].get(c, []):
                            table[(i, j)].setdefault(a, []).append((p * pb * pc, (a, tb, tc)))
            if not keep_all:                 # Viterbi: keep only the best per label
                for a in table[(i, j)]:
                    table[(i, j)][a] = [max(table[(i, j)][a], key=lambda x: x[0])]
    return table[(0, n)].get('S', [])

words = "astronomers saw stars with ears".split()
print('All parses:')
for p, t in sorted(probabilistic_cky(words, keep_all=True), reverse=True):
    print(f'  P = {p:.7f}   {bracket(t)}')
p, t = probabilistic_cky(words)[0]
print(f'\nMost probable (Viterbi): P = {p:.7f}\n{bracket(t)}')

All parses:
  P = 0.0009072   [S [NP astronomers] [VP [V saw] [NP [NP stars] [PP [P with] [NP ears]]]]]
  P = 0.0006804   [S [NP astronomers] [VP [VP [V saw] [NP stars]] [PP [P with] [NP ears]]]]

Most probable (Viterbi): P = 0.0009072
[S [NP astronomers] [VP [V saw] [NP [NP stars] [PP [P with] [NP ears]]]]]


In [ ]:
# The same with NLTK's ViterbiParser
from nltk import PCFG
from nltk.parse import ViterbiParser

pcfg = PCFG.fromstring("""
S -> NP VP [1.0]
PP -> P NP [1.0]
VP -> V NP [0.7] | VP PP [0.3]
NP -> NP PP [0.4] | 'astronomers' [0.1] | 'ears' [0.18] | 'saw' [0.04] | 'stars' [0.18] | 'telescopes' [0.1]
P -> 'with' [1.0]
V -> 'saw' [1.0]
""")
for tree in ViterbiParser(pcfg).parse("astronomers saw stars with ears".split()):
    print(tree, tree.prob())

### PCFG deficiencies
- **Independence assumption:** the expansion of a non-terminal is assumed independent of its context, which is often not true in natural language (e.g. subject NPs are more often pronouns than object NPs).
- **No lexical conditioning:** probabilities depend only on the non-terminals, not on the actual **words** in those positions (e.g. whether a PP attaches to the verb depends heavily on the verb and the preposition).

**Solution: Lexicalized PCFGs (L-PCFGs):** each production is associated with a **head word**, the central word that determines the syntactic type of the phrase.
e.g. `NP(head:dog) → Det(the) N(head:dog)`

# Dependency Grammar
- The syntactic structure of a sentence is described **solely** by **directed binary grammatical relations between words**.
- Words are connected by **directed links**.
- The **verb** is the center of the clause structure; every other unit is connected to it, directly or indirectly.

## Dependency parsing
Each relationship:
- Has one **head** and one **dependent** that modifies the head (arrow from head → dependent). e.g. *black ← car* with label **amod** (adjectival modifier).
- Is **labeled** by the type of dependency ([Universal Dependency relations](https://universaldependencies.org/u/dep/)): nsubj, obj, det, amod, prep/case, pobj/obl, punct, …

### Tree constraints
- Add a fake **ROOT** at the beginning so every word has a head; **only one** word is a dependent of ROOT (usually the main verb).
- Except for the root, each word has **exactly one incoming arc** (one head).
- There is a **unique path** from ROOT to each word.
- **No cycles** (A → B, B → C, C → A is not allowed).

### Projectivity
- An arc head → dependent is **projective** if there is a path from the head to **every word between** the head and the dependent.
- A tree is projective if **all** of its arcs are projective.
- Interpretation: **no crossing arcs** when the words are laid out in order with all arcs drawn above them.
- Non-projectivity arises from long-distance dependencies or languages with flexible word order.

In [12]:
def check_dependency_tree(words, heads):
    """heads[i] = index of the head of word i (1-based words, 0 = ROOT)."""
    n = len(words)
    roots = [i for i in range(1, n + 1) if heads[i] == 0]
    print('single dependent of ROOT :', len(roots) == 1, [words[i-1] for i in roots])
    # one head per word is guaranteed by the representation; check reachability & cycles
    def path_to_root(i):
        seen = set()
        while i != 0:
            if i in seen: return None      # cycle
            seen.add(i); i = heads[i]
        return seen
    print('no cycles, all reach ROOT:', all(path_to_root(i) is not None for i in range(1, n + 1)))
    def dominates(h, d):                  # is there a path h -> ... -> d ?
        while d != 0:
            d = heads[d]
            if d == h: return True
        return h == 0
    non_proj = [(words[heads[d]-1] if heads[d] else 'ROOT', words[d-1]) for d in range(1, n + 1)
                if not all(dominates(heads[d], m) for m in range(min(heads[d], d) + 1, max(heads[d], d)))]
    print('projective               :', not non_proj, non_proj or '')

# "There is a nodule in the left lung ." (heads as produced by spaCy, see below)
words = ['There', 'is', 'a', 'nodule', 'in', 'the', 'left', 'lung', '.']
heads = [None, 2, 0, 4, 2, 4, 8, 8, 5, 2]
check_dependency_tree(words, heads)

print('\nA non-projective example: "A hearing is scheduled on the issue today"')
words2 = ['A', 'hearing', 'is', 'scheduled', 'on', 'the', 'issue', 'today']
heads2 = [None, 2, 4, 4, 0, 2, 7, 5, 4]    # "on the issue" modifies "hearing" across "is scheduled"
check_dependency_tree(words2, heads2)

single dependent of ROOT : True ['is']
no cycles, all reach ROOT: True
projective               : True 

A non-projective example: "A hearing is scheduled on the issue today"
single dependent of ROOT : True ['scheduled']
no cycles, all reach ROOT: True
projective               : False [('hearing', 'on')]


## Dependency parsing algorithms
1. **Transition-based** dependency parsing
2. **Graph-based** dependency parsing

### 1. Transition-based ("shift-reduce")
Motivated by **shift-reduce parsing** of programming languages (Aho and Ullman, 1972).

**Components:** a **buffer** (words still to read), a **stack** (words being processed), and the **parser**, which picks an action.
- **Initial:** stack = [root], buffer = [all words], no edges
- **Final:** buffer = [] (all words attached)

**Method:** process the sentence word by word from the buffer, temporarily place words on the stack, and at each step perform an **action**:

| Action | Stack view (s1 = top, s2 = below) | Buffer view (w1 = first word of buffer) |
|---|---|---|
| **Shift** | push w1 onto the stack | — |
| **Left-arc** | add edge(s1 → s2) (s1 is head); remove s2 | add edge(w1 → s1); pop the stack. Needs non-empty stack and buffer; s1 can't be root |
| **Right-arc** | add edge(s2 → s1) (s2 is head); remove s1 | add edge(s1 → w1); pop s1 and **replace w1 with s1** at the front of the buffer |

**Oracle:** a supervised **classifier** that learns a function from the current **configuration** to the next action.
- Labels: Shift, Left-arc, Right-arc
- Features: top of the stack, first words of the buffer, their POS tags, …
- Trained on a treebank by replaying the gold trees into action sequences (as the code below does).

In [13]:
def transition_parse(words, gold_heads, verbose=True):
    """Buffer-based arc-standard parser (slide version) driven by a gold 'oracle'.
    words: list of tokens (index 1..n), gold_heads[i] = head index of word i (0 = root)."""
    name = lambda i: 'root' if i == 0 else words[i-1]
    stack, buffer, edges = [0], list(range(1, len(words) + 1)), []
    def has_all_children(h):
        return all((h, d) in edges for d in range(1, len(words) + 1) if gold_heads[d] == h)
    show = lambda a: print(f"{'[' + ' '.join(map(name, stack)) + ']':34}{'[' + ', '.join(map(name, buffer)) + ']':60}  {a}")
    if verbose: print(f"{'Stack':34}{'Buffer':60}  Action (that led here)"); show('(initial)')
    while buffer and not (stack == [] and buffer == [0]):
        s1, w1 = (stack[-1] if stack else None), buffer[0]
        if s1 not in (None, 0) and gold_heads[s1] == w1:
            action = 'Left-arc';  edges.append((w1, s1)); stack.pop()
        elif s1 is not None and gold_heads[w1] == s1 and has_all_children(w1):
            action = 'Right-arc'; edges.append((s1, w1)); stack.pop(); buffer[0] = s1
        else:
            action = 'Shift';     stack.append(buffer.pop(0))
        if verbose: show(action)
    return [(name(h), name(d)) for h, d in edges]

words = "Economic news had little effect on financial markets .".split()
#            Economic news had little effect on financial markets .
gold = [None,    2,     3,  0,   5,     3,    5,    8,       6,     3]
edges = transition_parse(words, gold)
print('\nEdges (head -> dependent):')
for h, d in edges: print(f'  {h} -> {d}')

Stack                             Buffer                                                        Action (that led here)
[root]                            [Economic, news, had, little, effect, on, financial, markets, .]  (initial)
[root Economic]                   [news, had, little, effect, on, financial, markets, .]        Shift
[root]                            [news, had, little, effect, on, financial, markets, .]        Left-arc
[root news]                       [had, little, effect, on, financial, markets, .]              Shift
[root]                            [had, little, effect, on, financial, markets, .]              Left-arc
[root had]                        [little, effect, on, financial, markets, .]                   Shift
[root had little]                 [effect, on, financial, markets, .]                           Shift
[root had]                        [effect, on, financial, markets, .]                           Left-arc
[root had effect]                 [on, financial

### 2. Graph-based
For a sentence like *"John saw Mary"*:
- **Nodes:** each word is a node in the graph.
- **Edges:** directed edges between pairs of words, from **head** (governor) to **dependent** (modifier), each with a **score**.

**Goal:** find the **most probable (highest-scoring) dependency tree**: connected, acyclic, with exactly one node with no incoming edge, the root (usually the main verb).

**Algorithms:**
- **Eisner's algorithm**: efficient dynamic programming for **projective** parsing (edges don't cross).
- **Chu-Liu/Edmonds' algorithm**: finds the **maximum spanning tree** in a directed graph; used for **non-projective** parsing (edges may cross).

In [14]:
from itertools import product

# Brute-force maximum spanning tree over a tiny scored graph (Chu-Liu/Edmonds does this efficiently)
words = ['ROOT', 'John', 'saw', 'Mary']
score = {(0, 1): 9, (0, 2): 10, (0, 3): 9,
         (2, 1): 30, (2, 3): 30, (1, 2): 20, (3, 2): 0, (1, 3): 3, (3, 1): 11}

def is_tree(heads):
    if list(heads.values()).count(0) != 1: return False
    for d in heads:                          # every word must reach ROOT without a cycle
        seen, x = set(), d
        while x != 0:
            if x in seen: return False
            seen.add(x); x = heads[x]
    return True

best = None
for choice in product(range(len(words)), repeat=len(words) - 1):
    heads = {d: h for d, h in zip(range(1, len(words)), choice)}
    if any(h == d for d, h in heads.items()) or not all((h, d) in score for d, h in heads.items()): continue
    if is_tree(heads):
        s = sum(score[(h, d)] for d, h in heads.items())
        if best is None or s > best[0]: best = (s, heads)

s, heads = best
print('best tree score:', s)
for d, h in heads.items(): print(f'  {words[h]} -> {words[d]}')

best tree score: 70
  saw -> John
  ROOT -> saw
  saw -> Mary


## Constituency ↔ Dependency
**Constituency → dependency** (via head words), for *"The quick brown fox jumps over the lazy dog"*:
- "jumps" is the main verb → **root**
- "fox" (head of the first NP) is the **subject** of "jumps"
- "over" (head of the PP) depends on "jumps"; "dog" (head of the second NP) is the object of "over"
- "quick", "brown" → dependents of "fox"; "lazy" → dependent of "dog"; each "the" → dependent of its noun

**Dependency → constituency:**
- The verb "jumps" is the main action → a **VP**
- "fox" + its modifiers → an **NP**
- "over the lazy dog" → a **PP**, with "dog" + modifiers forming another NP
- The sentence follows **S → NP VP**

In [15]:
# Constituency -> dependency using simple head rules
HEAD_RULES = {'S': 'VP', 'VP': 'V', 'NP': 'N', 'PP': 'P'}   # which child carries the head

def to_dependencies(tree, deps):
    """Return the head word of `tree`; append (head, dependent) arcs to deps."""
    label, *kids = tree
    if isinstance(kids[0], str):
        return kids[0]
    kid_heads = [(k[0], to_dependencies(k, deps)) for k in kids]
    head = next(h for lab, h in kid_heads if lab == HEAD_RULES[label])
    deps += [(head, h) for lab, h in kid_heads if h != head]
    return head

fox = ('S', ('NP', ('Det', 'The'), ('Adj', 'quick'), ('Adj', 'brown'), ('N', 'fox')),
            ('VP', ('V', 'jumps'), ('PP', ('P', 'over'), ('NP', ('Det', 'the'), ('Adj', 'lazy'), ('N', 'dog')))))
deps = []
root = to_dependencies(fox, deps)
print('ROOT ->', root)
for h, d in deps: print(f'{h} -> {d}')

ROOT -> jumps
fox -> The
fox -> quick
fox -> brown
dog -> the
dog -> lazy
over -> dog
jumps -> over
jumps -> fox


# Syntactic Parsing: Methods
### 1. Rule-based parsing
Uses **hand-crafted rules**, based on the grammar of the language and expressed in a formal grammar such as a CFG. *(optional detail)*
- **Recursive descent parser**: a **top-down** parser that recursively calls itself to match the input with grammar rules. Efficient and easy to implement, but hard to write for complex grammars (and loops on left recursion).
- **Shift-reduce parser**: bottom-up; shifts words onto a stack and reduces them with rules.
- **Chart parser**: **bottom-up**; builds a chart of **all possible partial parses** (like CKY). More complex, but handles complex grammars and ambiguous sentences.
- **Regexp parser**: uses **regular expressions** over tags to match patterns. Less expressive than CFGs (used for chunking).

### 2. Statistical parsing
Uses statistical models trained on a large corpus, learning to associate words with grammatical roles from past occurrences. Two families: **discriminative** and **generative**.
- **Hidden Markov Models (HMMs)**: POS tags are **hidden states**, words are **observed** symbols.
- **PCFGs**: CFGs with a probability for each rule.
- **Maximum Entropy Models (MEMs)**: can capture long-range dependencies; often used with PCFGs to improve accuracy.

### 3. Neural parsing (data-driven) *[covered LATER]*
Neural networks trained on large corpora learn to parse from input/output representations: RNNs, CNNs, attention mechanisms & Transformers, graph-based neural models.

In [ ]:
# NLTK rule-based parsers on the L1 grammar
import nltk
l1 = nltk.CFG.fromstring("""
S -> NP VP | Aux NP VP | VP
NP -> Pronoun | ProperNoun | Det Nominal
Nominal -> Noun | Nominal Noun | Nominal PP
VP -> Verb | Verb NP | Verb NP PP | Verb PP | VP PP
PP -> Preposition NP
Det -> 'that' | 'this' | 'the' | 'a'
Noun -> 'book' | 'flight' | 'meal' | 'money'
Verb -> 'book' | 'include' | 'prefer'
Pronoun -> 'I' | 'she' | 'me'
ProperNoun -> 'Houston' | 'NWA'
Aux -> 'does'
Preposition -> 'from' | 'to' | 'on' | 'near' | 'through'
""")
sent = "book that flight".split()

print('Chart parser (bottom-up):')
for t in nltk.ChartParser(l1).parse(sent): print(t)

# NOTE: RecursiveDescentParser would loop forever on the left-recursive rules
# (Nominal -> Nominal Noun, VP -> VP PP); try it on a grammar without them.

# Syntactic Parsing in Python (spaCy)
### POS tagging with spaCy
```
There --> PRON    is --> VERB    a --> DET    nodule --> NOUN
in --> ADP    the --> DET    left --> ADJ    lung --> NOUN    . --> PUNCT
```
spaCy's `token.pos_` uses the **Universal** POS tags (NOUN, VERB, ADP…); `token.tag_` gives the Penn Treebank tags (NN, VBZ, IN…).

In [ ]:
# !pip install spacy
# !python -m spacy download en_core_web_sm
import spacy
from spacy import displacy

# Load the language model
nlp = spacy.load("en_core_web_sm")

sentence = 'There is a nodule in the left lung.'

# nlp returns a Doc with per-token linguistic features and relationships
doc = nlp(sentence)

# Iterate over the tokens: print each token and its part-of-speech tag
for token in doc:
    print(token.text, "-->", token.pos_, f"({token.tag_})")

### Dependency parsing with spaCy
Expected output:
```
Token           | Relation | Head            | Children
----------------------------------------------------------------------
There           | expl     | is              | []
is              | ROOT     | is              | [There, nodule, .]
a               | det      | nodule          | []
nodule          | attr     | is              | [a, in]
in              | prep     | nodule          | [lung]
the             | det      | lung            | []
left            | amod     | lung            | []
lung            | pobj     | in              | [the, left]
.               | punct    | is              | []
```

In [ ]:
print("{:<15} | {:<8} | {:<15} | {:<20}".format('Token', 'Relation', 'Head', 'Children'))
print("-" * 70)

for token in doc:
    # Print the token, dependency nature, head and all dependents of the token
    print("{:<15} | {:<8} | {:<15} | {:<20}"
          .format(str(token.text), str(token.dep_), str(token.head.text), str([child for child in token.children])))

# Use displaCy to visualize the dependency
displacy.render(doc, style='dep', jupyter=True, options={'distance': 120})

# Shallow Parsing / Chunking
**Definition:** extracting short phrases (**chunks**) from a sentence.
- Unlike full parsing, chunking focuses on specific pieces, such as **noun phrases** or verb phrases, **without** analyzing the entire sentence structure.
- Also called **shallow parsing**.

*"The quick brown fox jumped over the lazy dog"* → noun phrases: **[The quick brown fox]NP**, **[the lazy dog]NP**

A common rule for NP chunks: `NP: {<DT>?<JJ>*<NN.*>+}` (optional determiner, any adjectives, one or more nouns). This is a regular expression over **POS tags** (a *Regexp parser*).

In [16]:
# NP chunking = a regex over the POS-tag sequence (tags given here so the cell runs without NLTK)
tagged = [('The', 'DT'), ('quick', 'JJ'), ('brown', 'JJ'), ('fox', 'NN'), ('jumped', 'VBD'),
          ('over', 'IN'), ('the', 'DT'), ('lazy', 'JJ'), ('dog', 'NN')]

tag_string = ''.join(f'<{t}>' for _, t in tagged)
np_rule = re.compile(r'(<DT>)?(<JJ>)*(<NN\w*>)+')

for m in np_rule.finditer(tag_string):
    start = tag_string[:m.start()].count('<')
    length = m.group().count('<')
    print('[' + ' '.join(w for w, _ in tagged[start:start + length]) + ']NP')

[The quick brown fox]NP
[the lazy dog]NP


In [ ]:
# Same with NLTK's RegexpParser
import nltk
chunker = nltk.RegexpParser("NP: {<DT>?<JJ>*<NN.*>+}")
tree = chunker.parse(nltk.pos_tag(nltk.word_tokenize("The quick brown fox jumped over the lazy dog")))
print(tree)
print([' '.join(w for w, t in st.leaves()) for st in tree.subtrees() if st.label() == 'NP'])

# spaCy gives noun chunks for free
print([chunk.text for chunk in nlp("The quick brown fox jumped over the lazy dog").noun_chunks])

# Semantic Parsing
**Definition:** translating natural language into a **formal meaning representation** that a machine can act on for some domain-specific application.
- Goes beyond syntactic parsing (grammatical structure) to capture the **underlying intent and meaning**.

### Example: radiology report → structured record
```
CHEST CT
11/03/2021
There is a 3mm nodule in the left lower lobe of lung.
```
→
```json
{"event": "CHEST CT", "date": "11/03/2021", "finding": "nodule",
 "attribute_type": "size", "attribute_value": "3mm"}
```

## Algorithms
### Rule-based
| Field | Technique |
|---|---|
| event | keyword matching |
| date | pattern matching |
| finding | keyword matching |
| attribute_type | pattern matching |
| attribute_value | pattern matching |

In [17]:
import json

report = """CHEST CT
11/03/2021
There is a 3mm nodule in the left lower lobe of lung."""

EVENTS   = ['CHEST CT', 'HEAD CT', 'ABDOMINAL CT', 'MRI', 'X-RAY']      # keyword lists
FINDINGS = ['nodule', 'mass', 'lesion', 'effusion', 'hemorrhage']
ATTRS    = [('size', r'\b(\d+(?:\.\d+)?\s?(?:mm|cm))\b')]                # (type, pattern)

def rule_based_semantic_parse(text):
    out = {}
    out['event']   = next((e for e in EVENTS if e.lower() in text.lower()), None)          # keyword
    m = re.search(r'\b(\d{1,2}/\d{1,2}/\d{4})\b', text)                                     # pattern
    out['date']    = m.group(1) if m else None
    out['finding'] = next((f for f in FINDINGS if re.search(rf'\b{f}\b', text, re.I)), None)  # keyword
    for attr_type, pat in ATTRS:                                                            # pattern
        m = re.search(pat, text)
        if m:
            out['attribute_type'], out['attribute_value'] = attr_type, m.group(1)
    return out

print(json.dumps(rule_based_semantic_parse(report), indent=2))

{
  "event": "CHEST CT",
  "date": "11/03/2021",
  "finding": "nodule",
  "attribute_type": "size",
  "attribute_value": "3mm"
}


### Statistical or neural network-based
- **Discriminative** algorithms directly predict the most likely semantic representation for a sentence, e.g. support vector machines, neural networks.
- **Generative** algorithms first generate a set of possible semantic representations and then assign probabilities to each, e.g. HMMs, PCFGs.
- The model **learns a representation** (a template with slots), e.g.:
```
<EVENT>
<DATE>
There is a <ATTR_VAL> <ATTR_TYPE> in the left lower lobe of lung.
```

# References
- Manning & Schütze, *Foundations of Statistical Natural Language Processing*
- Jurafsky & Martin, *Speech and Language Processing*: Ch. 17 Context-Free Grammars and Constituency Parsing; Ch. 18 Dependency Parsing; Ch. 14 Statistical Constituency Parsing ([slp3](https://web.stanford.edu/~jurafsky/slp3/))
- Jose Luis Verdu-Mas et al., "A Comparison of PCFG Models", CoNLL-2000
- [Universal Dependencies relations](https://universaldependencies.org/u/dep/)